# Audit initial du datset "Global Electronics Retailer"

Ce notebook a pour objectif d'explorer et d'évaluer la qualité des données brutes du dataset **Global Electronics Retailer** avant leur exploitation dans PostgreSQL et Power BI. L'audit permettra notamment de vérifier la structure des fichiers, les types de données, les valeurs manquantes, les doublons, l'unicité des clés, l'intégrité des relations entre les tables ainsi que certaines règles de cohérence métier.

## 1. Audit de la table `Sales`

Nous commençons par la table `Sales`, qui constitue la table transactionnelle centrale du dataset, afin de vérifier que les données sont correctement chargées et correspondent à la structure décrite dans le dictionnaire de données.

### 1.1 Chargement et 1ère inspection des données

In [35]:
# Import de la bibliothèque pandas
import pandas as pd

In [36]:
# Chargement des données de ventes dans un DataFrame
sales = pd.read_csv("../data/raw/Sales.csv")

In [37]:
# Aperçu des premières lignes de la table Sales
sales.head()

,Order Number,Line Item,Order Date,Delivery Date,CustomerKey,StoreKey,ProductKey,Quantity,Currency Code
0,366000,1,1/1/2016,NaN,265598,10,1304,1,CAD
1,366001,1,1/1/2016,1/13/2016,1269051,0,1048,2,USD
2,366001,2,1/1/2016,1/13/2016,1269051,0,2007,1,USD
3,366002,1,1/1/2016,1/12/2016,266019,0,1106,7,CAD
4,366002,2,1/1/2016,1/12/2016,266019,0,373,1,CAD


<div style=" display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;
">

<strong>Observation :</strong> Le fichier <code>Sales.csv</code> est correctement chargé et contient les 9 variables attendues d'après le dictionnaire de données. Les premières lignes confirment également qu'une même commande peut comporter plusieurs lignes de produits.

</div>

### 1.2 Vérification de la structure et des dimensions

Cette étape permet de vérifier la structure réelle de la table `Sales` et d'en mesurer le volume. Nous contrôlons le nombre de lignes et de colonnes ainsi que les types de données détectés par Pandas afin d'identifier d'éventuelles incohérences avant de poursuivre l'audit.

In [38]:
# Vérification des dimensions de la table Sales
print(f"Nombre de lignes : {sales.shape[0]}")
print(f"Nombre de colonnes : {sales.shape[1]}")

Nombre de lignes : 62884
Nombre de colonnes : 9


In [39]:
# Vérification des colonnes et de leurs types
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 62884 entries, 0 to 62883
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Order Number   62884 non-null  int64
 1   Line Item      62884 non-null  int64
 2   Order Date     62884 non-null  str  
 3   Delivery Date  13165 non-null  str  
 4   CustomerKey    62884 non-null  int64
 5   StoreKey       62884 non-null  int64
 6   ProductKey     62884 non-null  int64
 7   Quantity       62884 non-null  int64
 8   Currency Code  62884 non-null  str  
dtypes: int64(6), str(3)
memory usage: 4.3 MB


<div style=" display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;
">

<strong>Observation :</strong> La table <code>Sales</code> contient 62 884 lignes de commande et 9 colonnes. Toutes les variables sont complètes à l'exception de `Delivery Date`, renseignée sur seulement 13 165 lignes. <code>Order Date</code> et <code>Delivery Date</code> sont par ailleurs interprétées comme des chaînes de caractères et devront faire l'objet d'une vérification avant leur utilisation comme variables temporelles.

</div>

### 1.3. Analyse des valeurs manquantes

Cette étape vise à identifier et quantifier les valeurs manquantes dans la table `Sales`. Leur analyse permet de déterminer si certaines variables sont incomplètes et d'évaluer si ces absences constituent une anomalie de qualité ou peuvent s'expliquer par une logique métier.

In [40]:
# Comptage des valeurs manquantes par colonne
sales.isna().sum()

Order Number         0
Line Item            0
Order Date           0
Delivery Date    49719
CustomerKey          0
StoreKey             0
ProductKey           0
Quantity             0
Currency Code        0
dtype: int64

In [41]:
# Calcul du pourcentage de valeurs manquantes par colonne
(sales.isna().mean() * 100).round(2)

Order Number      0.00
Line Item         0.00
Order Date        0.00
Delivery Date    79.06
CustomerKey       0.00
StoreKey          0.00
ProductKey        0.00
Quantity          0.00
Currency Code     0.00
dtype: float64

<div style=" display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;
">

<strong>Observation :</strong> La colonne <code>Delivery Date</code> est la seule variable contenant des valeurs manquantes, avec 49 719 valeurs absentes, soit 79,06 % des lignes de commande. Toutes les autres colonnes sont complètes. Ce taux élevé nécessite une investigation afin de déterminer si ces absences correspondent à une logique métier ou à un problème de qualité des données.

</div>

#### 1.3.1 Investigation des valeurs manquantes de `Delivery Date`

Le taux élevé de valeurs manquantes dans `Delivery Date` nécessite d'en rechercher l'origine. Nous vérifions dans un premier temps si ces absences sont associées à certains magasins (`StoreKey`), ce qui pourrait révéler une logique liée au mode de vente.

In [42]:
# Identification des StoreKey associés aux dates de livraison manquantes
sales.loc[sales["Delivery Date"].isna(), "StoreKey"].value_counts().sort_index()

StoreKey
1      292
2       18
4      431
5      892
6      615
8     1360
9     1577
10    1269
12     224
13     184
14     126
15     235
16     214
17     210
18     194
19     635
20     443
21     391
22     800
23     674
24     703
26     295
27     721
28     214
29    1011
30     976
31     417
32     454
33     498
34     448
36    1049
37     995
38    1034
39    1054
40    1050
41     444
42    1017
43    1340
44    1436
45    1471
47    1348
48    1289
49    1041
50    1519
51    1356
53    1284
54    1498
55    1518
56    1287
57    1442
59    1472
61    1485
62     778
63     948
64    1353
65    1395
66    1295
Name: count, dtype: int64

In [43]:
# Comparaison de la présence d'une date de livraison selon le StoreKey
pd.crosstab(
    sales["StoreKey"].eq(0),
    sales["Delivery Date"].isna(),
    rownames=["StoreKey = 0"],
    colnames=["Delivery Date manquante"]
)

Delivery Date manquante,False,True
StoreKey = 0,,
False,0,49719
True,13165,0


#### 1.3.2 Vérification de la signification de `StoreKey = 0`

La relation observée montre que toutes les lignes associées à `StoreKey = 0` possèdent une date de livraison, tandis que celles associées à un autre magasin n'en possèdent aucune. Nous consultons ponctuellement la table `Stores` afin de vérifier si `StoreKey = 0` correspond à un mode de vente particulier.

In [44]:
# Chargement ponctuel de la table Stores pour vérifier la signification de StoreKey = 0
stores = pd.read_csv("../data/raw/Stores.csv")

# Recherche des informations associées à StoreKey = 0
stores.loc[stores["StoreKey"] == 0]

,StoreKey,Country,State,Square Meters,Open Date
66,0,Online,Online,NaN,1/1/2010


<div style=" display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #F5BB27; background-color: #FFF8D6; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;
">

<strong>Conclusion de l'investigation :</strong> L'analyse croisée montre que les 49 719 valeurs manquantes de <code>Delivery Date</code> concernent exclusivement les ventes associées à un <code>StoreKey</code> différent de 0, tandis que les 13 165 lignes associées à <code>StoreKey = 0</code> disposent toutes d'une date de livraison. La table <code>Stores</code> confirme que <code>StoreKey = 0</code> correspond au canal de vente <strong>Online</strong>. L'absence de <code>Delivery Date</code> pour les autres magasins est donc cohérente avec la logique métier des ventes réalisées en magasin physique et ne constitue pas une anomalie de qualité des données.

</div>


<div style=" display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #0ba84f; background-color: #ddffe7; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;
">

<strong>Décision :</strong> Les valeurs manquantes de <code>Delivery Date</code> sont conservées, car elles correspondent à une absence de livraison attendue pour les ventes réalisées en magasin physique.

</div>

### 1.4. Vérification des doublons et de l'unicité des clés

Cette étape a comme objectif de détecter d'éventuelles lignes dupliquées et à vérifier l'unicité des identifiants de la table `Sales`. Nous cherchons notamment à confirmer si la combinaison `Order Number` et `Line Item` permet d'identifier de manière unique chaque ligne de commande.

In [45]:
# Vérification des lignes entièrement dupliquées (Doublon exact)
print(f'Le nombre des lignes entièrement dupliquées dans la table "Sales" est : {sales.duplicated().sum()}')

Le nombre des lignes entièrement dupliquées dans la table "Sales" est : 0


In [46]:
# Vérification de l'unicité des numéros de commande (ID métier de commande)
print(f"Nombre de lignes de commande : {len(sales)}")
print(f"Nombre de commandes distinctes : {sales['Order Number'].nunique()}")

Nombre de lignes de commande : 62884
Nombre de commandes distinctes : 26326


In [47]:
# Vérification de l'unicité de la clé candidate composée (Order Number, Line Item)
duplicate_keys = sales.duplicated(subset=["Order Number", "Line Item"]).sum()
print(f"Nombre de doublons sur la clé (Order Number, Line Item) : {duplicate_keys}")

Nombre de doublons sur la clé (Order Number, Line Item) : 0


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;">

<strong>Observation :</strong> La table <code>Sales</code> ne contient aucune ligne entièrement dupliquée. Les 62 884 lignes de commande correspondent à 26 326 commandes distinctes, confirmant qu'un même <code>Order Number</code> peut être associé à plusieurs lignes de produits. En revanche, aucune duplication n'est observée sur la combinaison <code>(Order Number, Line Item)</code>, ce qui confirme son unicité et permet de la considérer comme une clé candidate composée pour identifier chaque ligne de commande.

</div>

### 1.5. Vérification de la cohérence des valeurs

Au-delà de la structure des données, il est nécessaire de s'assurer que les valeurs enregistrées restent cohérentes avec leur signification métier. Nous recherchons ici d'éventuelles valeurs impossibles, inattendues ou hors domaine susceptibles d'affecter la fiabilité des analyses.

In [48]:
# Vérification des valeurs présentes dans la colonne Quantity
sales["Quantity"].value_counts().sort_index()

Quantity
1     18311
2     13611
3     10907
4      5441
5      3619
6      3699
7      3639
8      1849
9       895
10      913
Name: count, dtype: int64

In [49]:
# Vérification des valeurs présentes dans la colonne Currency Code
sales["Currency Code"].value_counts().sort_index()

Currency Code
AUD     2941
CAD     5415
EUR    12621
GBP     8140
USD    33767
Name: count, dtype: int64

In [50]:
# Chargement ponctuel de la table Exchange_Rates pour vérifier les devises disponibles
exchange_rates = pd.read_csv("../data/raw/Exchange_Rates.csv")

# Identification des devises disponibles dans la table Exchange_Rates
exchange_rates["Currency"].unique()

<StringArray>
['USD', 'CAD', 'AUD', 'EUR', 'GBP']
Length: 5, dtype: str

<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;">

<strong>Observation :</strong> Les quantités vendues sont toutes comprises entre 1 et 10, sans valeur nulle ou négative. La colonne <code>Currency Code</code> contient cinq devises distinctes (<code>AUD</code>, <code>CAD</code>, <code>EUR</code>, <code>GBP</code> et <code>USD</code>), toutes présentes dans la table de référence <code>Exchange_Rates</code>. Aucune valeur incohérente n'a été identifiée sur les domaines contrôlés.

</div>

### 1.6. Analyse et cohérence des données temporelles

Les dates jouent un rôle central dans l'analyse des ventes, notamment pour étudier leur évolution, la saisonnalité ou les délais de livraison. Avant toute analyse temporelle, nous devons vérifier leur format, leur couverture chronologique et la cohérence entre les dates de commande et de livraison.

In [51]:
# Conversion des colonnes de dates au format datetime
order_date = pd.to_datetime(sales["Order Date"], format="%m/%d/%Y", errors="coerce")
delivery_date = pd.to_datetime(sales["Delivery Date"], format="%m/%d/%Y", errors="coerce")

# Vérification des dates de commande invalides après conversion
print(f"Nombre de dates de commande invalides : {order_date.isna().sum()}")

Nombre de dates de commande invalides : 0


In [52]:
# Vérification des dates de livraison invalides après conversion
missing_delivery_before = sales["Delivery Date"].isna().sum()
missing_delivery_after = delivery_date.isna().sum()

print(f"Valeurs manquantes avant conversion : {missing_delivery_before}")
print(f"Valeurs manquantes après conversion : {missing_delivery_after}")
print(f"Dates de livraison invalides : {missing_delivery_after - missing_delivery_before}")

Valeurs manquantes avant conversion : 49719
Valeurs manquantes après conversion : 49719
Dates de livraison invalides : 0


In [53]:
# Identification de la période couverte par les commandes
print(f"Première date de commande : {order_date.min().date()}")
print(f"Dernière date de commande : {order_date.max().date()}")

Première date de commande : 2016-01-01
Dernière date de commande : 2021-02-20


In [54]:
# Vérification de la répartition des lignes de commande par année
order_date.dt.year.value_counts().sort_index()

Order Date
2016     6905
2017     7942
2018    14188
2019    21611
2020    11026
2021     1212
Name: count, dtype: int64

In [55]:
# Vérification des livraisons antérieures à la date de commande
invalid_delivery_order = (delivery_date < order_date).sum()
print(f"Nombre de livraisons antérieures à la date de commande : {invalid_delivery_order}")

Nombre de livraisons antérieures à la date de commande : 0


In [56]:
# Calcul et synthèse des délais de livraison en jours
delivery_delay = (delivery_date - order_date).dt.days

print(f"Nombre de lignes livrées : {delivery_delay.count():,.0f}".replace(",", " "))
print(f"Délai moyen : {delivery_delay.mean():.2f}".replace(".", ",") + " jours")
print(f"Délai médian : {delivery_delay.median():.0f} jours")
print(f"1er quartile : {delivery_delay.quantile(0.25):.0f} jours")
print(f"3e quartile : {delivery_delay.quantile(0.75):.0f} jours")
print(f"Délai minimum : {delivery_delay.min():.0f} jour")
print(f"Délai maximum : {delivery_delay.max():.0f} jours")

Nombre de lignes livrées : 13 165
Délai moyen : 4,53 jours
Délai médian : 4 jours
1er quartile : 3 jours
3e quartile : 6 jours
Délai minimum : 1 jour
Délai maximum : 17 jours


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;">

<strong>Observation :</strong> Les dates de commande et les dates de livraison renseignées sont toutes conformes au format attendu, sans valeur invalide détectée. Les commandes couvrent la période du <strong>1er janvier 2016 au 20 février 2021</strong>, avec des observations présentes chaque année ; l'année 2021 est toutefois partielle. Aucune livraison n'est antérieure à sa date de commande. Pour les 13 165 lignes disposant d'une date de livraison, les délais sont compris entre 1 et 17 jours, avec une médiane de 4 jours et une moyenne d'environ 4,5 jours, sans incohérence temporelle manifeste.

</div>

### 1.7. Vérification de l'intégrité référentielle

La table `Sales` contient plusieurs clés permettant de relier chaque ligne de commande aux entités de référence du dataset. Nous vérifions ici que les identifiants présents dans `Sales` possèdent bien une correspondance dans les tables associées, afin de détecter d'éventuelles références orphelines.

In [57]:
# Chargement ponctuel de la table Customers pour vérifier les références clients
customers = pd.read_csv("../data/raw/Customers.csv", encoding="cp1252")

In [58]:
# Identification des CustomerKey de Sales absents de la table Customers
orphan_customer_keys = ~sales["CustomerKey"].isin(customers["CustomerKey"])
print(f"Nombre de lignes de commande avec un CustomerKey sans correspondance : {orphan_customer_keys.sum()}")

Nombre de lignes de commande avec un CustomerKey sans correspondance : 0


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> Tous les <code>CustomerKey</code> présents dans <code>Sales</code> possèdent une correspondance dans la table <code>Customers</code> ; aucune référence client orpheline n'a été détectée.</div>

In [59]:
# Chargement ponctuel de la table Products pour vérifier les références produits
products = pd.read_csv("../data/raw/Products.csv")

In [60]:
# Identification des ProductKey de Sales absents de la table Products
orphan_product_keys = ~sales["ProductKey"].isin(products["ProductKey"])
print(f"Nombre de lignes de commande avec un ProductKey sans correspondance : {orphan_product_keys.sum()}")

Nombre de lignes de commande avec un ProductKey sans correspondance : 0


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> Aucun <code>ProductKey</code> orphelin n'a été identifié : chaque produit référencé dans <code>Sales</code> existe bien dans la table <code>Products</code>.</div>

In [61]:
# La table Stores a été chargée précédemment dans le notebook
# Identification des StoreKey de Sales absents de la table Stores

orphan_store_keys = ~sales["StoreKey"].isin(stores["StoreKey"])
print(f"Nombre de lignes de commande avec un StoreKey sans correspondance : {orphan_store_keys.sum()}")

Nombre de lignes de commande avec un StoreKey sans correspondance : 0


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> L'ensemble des <code>StoreKey</code> utilisés dans <code>Sales</code> est référencé dans la table <code>Stores</code>, y compris la valeur <code>0</code> associée au canal de vente Online ; aucune référence magasin orpheline n'est donc présente.</div>

In [62]:
# La table Exchange_Rates a été chargée précédemment dans le notebook
# Conversion temporaire de sa colonne Date au format datetime

exchange_date = pd.to_datetime(
    exchange_rates["Date"],
    format="%m/%d/%Y",
    errors="coerce"
)

print(f"Nombre de dates invalides dans Exchange_Rates : {exchange_date.isna().sum()}")

Nombre de dates invalides dans Exchange_Rates : 0


In [63]:
# Vérification de la correspondance entre Sales et Exchange_Rates sur la date et la devise

sales_exchange_check = pd.DataFrame({
    "Order Date": order_date,
    "Currency Code": sales["Currency Code"]
})

exchange_check = pd.DataFrame({
    "Date": exchange_date,
    "Currency": exchange_rates["Currency"]
})

sales_exchange_check = sales_exchange_check.merge(
    exchange_check,
    how="left",
    left_on=["Order Date", "Currency Code"],
    right_on=["Date", "Currency"],
    indicator=True
)

orphan_exchange_rates = (sales_exchange_check["_merge"] == "left_only").sum()

print(f"Nombre de lignes de commande sans taux de change correspondant : {orphan_exchange_rates}")

Nombre de lignes de commande sans taux de change correspondant : 0


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> Chaque combinaison <code>(Order Date, Currency Code)</code> utilisée dans <code>Sales</code> trouve une correspondance sur <code>(Date, Currency)</code> dans <code>Exchange_Rates</code> ; aucune ligne de commande ne présente donc de référence orpheline vis-à-vis des taux de change.</div>

<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #F5BB27; background-color: #FFF8D6; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Conclusion de l'investigation :</strong> Les contrôles d'intégrité référentielle ne révèlent aucune référence orpheline dans <code>Sales</code> : les relations avec <code>Customers</code>, <code>Products</code>, <code>Stores</code> et <code>Exchange_Rates</code> sont toutes cohérentes avec le modèle de données attendu.</div>

### 1.8. Vérification de la cohérence de Line Item

Après avoir confirmé l'unicité de la combinaison `(Order Number, Line Item)`, nous examinons la numérotation des lignes au sein de chaque commande. Ce contrôle exploratoire repose sur l'hypothèse que les `Line Item` suivent une séquence continue à partir de 1 ; cette règle n'étant pas explicitement documentée dans le dictionnaire de données, les éventuelles ruptures détectées devront être investiguées avant d'être considérées comme des anomalies.

In [64]:
# Vérification des valeurs présentes dans la colonne Line Item
sales["Line Item"].value_counts().sort_index()

Line Item
1    26326
2    16789
3    10248
4     5125
5     2175
6     1476
7      745
Name: count, dtype: int64

In [65]:
# Vérification de la continuité des Line Item au sein de chaque commande

line_item_check = sales.groupby("Order Number")["Line Item"].agg(["count", "max"])

invalid_line_item_sequences = (
    line_item_check["count"] != line_item_check["max"]
).sum()

print(f"Nombre de commandes avec une séquence de Line Item incohérente : {invalid_line_item_sequences}")

Nombre de commandes avec une séquence de Line Item incohérente : 621


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> Le contrôle exploratoire de la numérotation des <code>Line Item</code> identifie <strong>621 commandes</strong> pour lesquelles le nombre de lignes diffère du numéro de <code>Line Item</code> maximal, suggérant une possible rupture dans la séquence attendue <code>1...n</code>. Ce résultat nécessite une investigation avant de pouvoir être interprété comme une anomalie.</div>

#### 1.8.1. Investigation des ruptures de séquence de `Line Item`

Le contrôle exploratoire a identifié 621 commandes pour lesquelles le nombre de lignes diffère du numéro de `Line Item` maximal. Nous examinons ces commandes afin de comprendre l'origine de ces écarts et de déterminer s'ils correspondent à de véritables anomalies ou à une particularité de la structure des données.

In [66]:
# Synthèse des premières commandes présentant une rupture de séquence
line_item_investigation = (
    sales.groupby("Order Number")["Line Item"]
    .agg(
        nombre_lignes="count",
        line_item_max="max",
        line_items=lambda x: sorted(x.tolist())
    )
)

line_item_investigation = line_item_investigation[
    line_item_investigation["nombre_lignes"] != line_item_investigation["line_item_max"]
]

line_item_investigation.head(10)

,nombre_lignes,line_item_max,line_items
Order Number,,,
372008,2,3,"[1, 3]"
374000,5,6,"[1, 2, 3, 5, 6]"
384000,3,4,"[1, 2, 4]"
385009,2,3,"[1, 3]"
390001,5,6,"[1, 2, 3, 4, 6]"
398003,3,4,"[1, 3, 4]"
409011,5,6,"[1, 2, 3, 4, 6]"
411006,3,4,"[1, 2, 4]"
420005,5,6,"[1, 2, 4, 5, 6]"


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> Sur les 10 premiers cas, j'observe quelque chose : il semble manquer exactement un numéro à chaque fois. Vérifions si ceci est le cas pour toutes les 621 commandes ou bien il y'a des exceptions.</div>

In [67]:
# Calcul du nombre de Line Item manquants pour chaque commande à investiguer

line_item_investigation["nombre_manquants"] = (
    line_item_investigation["line_item_max"]
    - line_item_investigation["nombre_lignes"]
)

line_item_investigation["nombre_manquants"].value_counts().sort_index()

nombre_manquants
1    612
2      8
3      1
Name: count, dtype: int64

<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #f33e11; background-color: #ffe5e5; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Observation :</strong> Les 612 cas avec un seul trou semblent constituer le comportement dominant avec 612 commandes. Les 9 autres peuvent nous aider à comprendre si on observe simplement le même phénomène plusieurs fois ou quelque chose de différent. Vérifions les 9 cas le splus atypiques càd ceux où il manque 2 ou 3 numéros.</div>

In [68]:
# Affichage des commandes présentant plusieurs Line Item manquants

line_item_investigation[
    line_item_investigation["nombre_manquants"] > 1
]

,nombre_lignes,line_item_max,line_items,nombre_manquants
Order Number,,,,
783003,4,6,"[1, 2, 4, 6]",2
906002,3,5,"[1, 3, 5]",2
909001,4,6,"[1, 3, 5, 6]",2
1014001,5,7,"[1, 2, 3, 5, 7]",2
1233014,5,7,"[1, 2, 4, 5, 7]",2
1506001,4,6,"[1, 3, 5, 6]",2
1508054,4,7,"[1, 3, 5, 7]",3
1807023,5,7,"[1, 2, 3, 5, 7]",2
1851013,5,7,"[1, 2, 4, 5, 7]",2


<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #F5BB27; background-color: #FFF8D6; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Conclusion de l'investigation :</strong> Les 621 commandes identifiées présentent bien des ruptures dans la numérotation des <code>Line Item</code>. Parmi elles, 612 comportent un seul numéro manquant, 8 en comportent deux et 1 en comporte trois. L'examen des cas les plus atypiques confirme qu'il s'agit du même phénomène de discontinuité de numérotation. Toutefois, le dictionnaire de données indique uniquement que <code>Line Item</code> identifie les produits au sein d'une commande et ne précise pas que leur numérotation doit obligatoirement former une séquence continue. Ces ruptures ne peuvent donc pas être considérées comme des anomalies de qualité sur la seule base de ce contrôle.</div>

<div style="display: block; width: 100%; max-width: 100%; box-sizing: border-box; border-left: 5px solid #0ba84f; background-color: #ddffe7; padding: 12px 16px; border-radius: 4px; white-space: normal; overflow-wrap: break-word;"><strong>Décision :</strong> Les valeurs de <code>Line Item</code> sont conservées telles quelles. En l'absence de règle métier imposant une numérotation continue, aucune correction ne sera appliquée. La combinaison <code>(Order Number, Line Item)</code> reste néanmoins valide comme clé candidate puisqu'elle est unique pour l'ensemble des lignes de la table <code>Sales</code>.</div>

### Bilan de l'audit de la table `Sales`

L'audit de la table `Sales`, composée de **62 884 lignes de commande réparties sur 26 326 commandes**, n'a révélé aucune anomalie majeure susceptible de compromettre son exploitation analytique.

Les principaux contrôles ont permis de constater que :

- la structure de la table est cohérente avec le dictionnaire de données et aucune ligne entièrement dupliquée n'a été détectée ;
- la combinaison `(Order Number, Line Item)` est unique sur l'ensemble des données et constitue une clé candidate permettant d'identifier chaque ligne de commande ;
- les valeurs de `Quantity` sont positives et comprises entre 1 et 10, tandis que les cinq devises utilisées dans `Sales` sont toutes référencées dans `Exchange_Rates` ;
- `Delivery Date` présente 49 719 valeurs manquantes (79,06 %), mais l'investigation a montré qu'elles concernent exclusivement les ventes réalisées en magasin physique. Les ventes `Online` disposent toutes d'une date de livraison : ces valeurs manquantes sont donc cohérentes avec la logique métier et ne nécessitent aucune correction ;
- les dates renseignées sont valides et chronologiquement cohérentes. Les commandes couvrent la période du **1er janvier 2016 au 20 février 2021**, avec une année 2021 partielle à prendre en compte dans les futures analyses temporelles ;
- aucune référence orpheline n'a été détectée entre `Sales` et les tables `Customers`, `Products`, `Stores` et `Exchange_Rates` ;
- 621 commandes présentent des ruptures dans la numérotation des `Line Item`. Toutefois, aucune règle métier ne précisant que cette numérotation doit être continue, ces cas sont conservés tels quels et ne sont pas considérés comme des anomalies.

La table `Sales` présente ainsi un niveau de qualité satisfaisant pour la poursuite du projet. Les particularités identifiées devront néanmoins être prises en compte lors des prochaines étapes de préparation, de modélisation et d'analyse des données.